In [1]:
from nanover.openmm import OpenMMSimulation
from nanover.mdanalysis.converter import frame_data_to_mdanalysis

simulation = OpenMMSimulation.from_xml_path("../ase/openmm_files/neuraminidase.xml")
simulation.load()
universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

In [2]:
from nanover.app import OmniRunner

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="MOVEABLE RESTRAINTS")
imd_runner.load(0)
imd = imd_runner.app_server.imd

In [ ]:
from nanover.utilities.change_buffers import DictionaryChange
from nanover.imd.imd_state import INTERACTION_PREFIX, ParticleInteraction

MOVEABLE_RESTRAINTS: dict[str, ParticleInteraction] = {}
ACTIVE_RESTRAINTS: set[str] = set()

def add_restraint(indexes):
    key = f"{INTERACTION_PREFIX}MOVEABLE-RESTRAINT.{len(MOVEABLE_RESTRAINTS)}"
    restraint = ParticleInteraction((0, 0, 0), indexes)
    restraint.scale = 5000
    restraint.interaction_type = "spring"
    restraint.reset_velocities = True
    MOVEABLE_RESTRAINTS[key] = restraint
    return key


def enable_restraint(key):
    if key in ACTIVE_RESTRAINTS:
        return
    ACTIVE_RESTRAINTS.add(key)
    print("ENABLE", key)
    positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
    restraint = MOVEABLE_RESTRAINTS[key]
    restraint.position = positions[restraint.particles].sum(axis=0) / len(restraint.particles)
    imd.insert_interaction(key, restraint)


def disable_restraint(key):
    if key not in ACTIVE_RESTRAINTS:
        return
    print("DISABLE", key)
    ACTIVE_RESTRAINTS.remove(key)
    imd.remove_interaction(key)


def check_restraints():
    interacted_indexes = {
        index
        for key, interaction in imd.active_interactions.items() if "MOVEABLE-RESTRAINT" not in key
        for index in interaction.particles
    }

    for key, restraint in MOVEABLE_RESTRAINTS.items():
        if interacted_indexes.intersection(restraint.particles):
            disable_restraint(key)
        else:
            enable_restraint(key)


def on_state_changed(change: DictionaryChange, **kwargs):
    relevant = (any(key.startswith(INTERACTION_PREFIX) and "MOVEABLE-RESTRAINT" not in key for key in change.updates)
        or any(key.startswith(INTERACTION_PREFIX) and "MOVEABLE-RESTRAINT" not in key for key in change.removals))
    if relevant:
        check_restraints()


imd_runner.app_server.state_dictionary.content_updated.add_callback(on_state_changed)

In [ ]:
ligand = universe.select_atoms("chainID B")
add_restraint(list(map(int, ligand.indices)))

In [3]:
from nanover.websocket import NanoverImdClient

client = NanoverImdClient.from_runner(imd_runner)
ligand_sel = client.create_selection("ligand", map(int, ligand.indices))
with ligand_sel.modify():
    ligand_sel.renderer = {
            'color': 'blue',
            'scale': 0.1,
            'render': 'liquorice'
        }
    ligand_sel.velocity_reset = True
    ligand_sel.interaction_method = 'group'

In [ ]:
TEST_INTERACTION = INTERACTION_PREFIX + "test"
imd.insert_interaction(TEST_INTERACTION, ParticleInteraction((10, 10, 10), list(ligand._selected_particle_ids), scale=100, interaction_type="spring", reset_velocities=True))

In [ ]:
imd.remove_interaction(TEST_INTERACTION)

In [ ]:
imd_runner.app_server.imd.active_interactions